In [4]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("03_Customer_Churn_Prediction_and_Retention_Decision_System").resolve().parents[2]
DATA_RAW = PROJECT_ROOT / "data" / "raw"
RAW_DATA_FILE = DATA_RAW / "Telco_customer_churn.xlsx"

In [7]:
df = pd.read_excel(RAW_DATA_FILE)

In [ ]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [9]:
df.shape

(7043, 33)

In [11]:
df.dtypes

CustomerID            object
Count                  int64
Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
Churn Label           object
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason          object
dtype: object

In [15]:
clean_df = df.copy()

In [16]:
clean_df.columns = (
        clean_df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

In [17]:
clean_df.columns.tolist()

['customerid',
 'count',
 'country',
 'state',
 'city',
 'zip_code',
 'lat_long',
 'latitude',
 'longitude',
 'gender',
 'senior_citizen',
 'partner',
 'dependents',
 'tenure_months',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'monthly_charges',
 'total_charges',
 'churn_label',
 'churn_value',
 'churn_score',
 'cltv',
 'churn_reason']

In [18]:
missing_summary = pd.DataFrame({
        "missing_count": clean_df.isna().sum(),
        "missing_pct": (clean_df.isna().sum() / len(clean_df) * 100).round(2)
    }).sort_values(by="missing_pct", ascending=False)

In [21]:
missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_pct
churn_reason,5174,73.46


In [22]:
clean_df.duplicated().sum()

0

In [38]:
unique_summary = pd.DataFrame({
        "n_unique": clean_df.nunique(dropna=False),
        "dtype": clean_df.dtypes.astype(str)
    }).sort_values(by="n_unique", ascending=False)

unique_summary

,n_unique,dtype
customerid,7043,object
total_charges,6531,object
cltv,3438,int64
zip_code,1652,int64
lat_long,1652,object
latitude,1652,float64
longitude,1651,float64
monthly_charges,1585,float64
city,1129,object
churn_score,85,int64


In [34]:
leakage_candidates = [
        "customerid", "customer_id",
        "count",
        "country",
        "state",
        "churn_label", "churn_value",
        "churn_score",
        "cltv",
        "churn_reason"
    ]

print("\n=== REVIEW COLUMNS (LEAKAGE / LOW VALUE / ID) ===")
print([col for col in leakage_candidates if col in clean_df.columns])


=== REVIEW COLUMNS (LEAKAGE / LOW VALUE / ID) ===
['customerid', 'count', 'country', 'state', 'churn_label', 'churn_value', 'churn_score', 'cltv', 'churn_reason']


In [39]:
if "churn_value" in clean_df.columns:
        print("\n=== TARGET DISTRIBUTION: churn_value ===")
        print('Churn1 ' + str(clean_df["churn_value"].value_counts(dropna=False).get(1, 0)))
        print('Churn0 ' + str(clean_df["churn_value"].value_counts(dropna=False).get(0, 0)))
        print('churn2 ' + str(clean_df["churn_value"].value_counts(normalize=True, dropna=False).round(4)))
elif "churn_label" in clean_df.columns:
        print("\n=== TARGET DISTRIBUTION: churn_label ===")
        print(clean_df["churn_label"].value_counts(dropna=False))
        print(clean_df["churn_label"].value_counts(normalize=True, dropna=False).round(4))


=== TARGET DISTRIBUTION: churn_value ===
Churn1 1869
Churn0 5174
churn2 churn_value
0    0.7346
1    0.2654
Name: proportion, dtype: float64


In [40]:
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [41]:
 df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

In [42]:
df.columns

Index(['customerid', 'count', 'country', 'state', 'city', 'zip_code',
       'lat_long', 'latitude', 'longitude', 'gender', 'senior_citizen',
       'partner', 'dependents', 'tenure_months', 'phone_service',
       'multiple_lines', 'internet_service', 'online_security',
       'online_backup', 'device_protection', 'tech_support', 'streaming_tv',
       'streaming_movies', 'contract', 'paperless_billing', 'payment_method',
       'monthly_charges', 'total_charges', 'churn_label', 'churn_value',
       'churn_score', 'cltv', 'churn_reason'],
      dtype='object')

In [44]:
object_cols = df.select_dtypes(include="object").columns.tolist()
object_cols

['customerid',
 'country',
 'state',
 'city',
 'lat_long',
 'gender',
 'senior_citizen',
 'partner',
 'dependents',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'total_charges',
 'churn_label',
 'churn_reason']

In [46]:
    print("\n=== BLANK / WHITESPACE CHECK IN OBJECT COLUMNS ===")
    blank_summary = []
    for col in object_cols:
        blank_count = df[col].astype(str).str.strip().eq("").sum()
        blank_summary.append({"column": col, "blank_count": blank_count})
    blank_summary_df = pd.DataFrame(blank_summary).sort_values(by="blank_count", ascending=False)
    print(blank_summary_df[blank_summary_df["blank_count"] > 0])


=== BLANK / WHITESPACE CHECK IN OBJECT COLUMNS ===
           column  blank_count
21  total_charges           11


In [48]:
    print("\n=== TOTAL_CHARGES RAW SAMPLE (TOP 10 DISTINCT VALUES) ===")
    if "total_charges" in df.columns:
        print(df["total_charges"].astype(str).str.strip().value_counts(dropna=False).head(10))

        df["total_charges_clean"] = pd.to_numeric(df["total_charges"].astype(str).str.strip(), errors="coerce")

        print("\n=== TOTAL_CHARGES CONVERSION CHECK ===")
        print("Original dtype:", df["total_charges"].dtype)
        print("Converted dtype:", df["total_charges_clean"].dtype)
        print("Nulls after conversion:", df["total_charges_clean"].isna().sum())

        print("\nRows where total_charges became NaN after conversion:")
        invalid_total_charges = df[df["total_charges_clean"].isna()][["customerid", "tenure_months", "monthly_charges", "total_charges", "churn_label", "churn_value"]]
        print(invalid_total_charges.head(20))


=== TOTAL_CHARGES RAW SAMPLE (TOP 10 DISTINCT VALUES) ===
total_charges
20.2     11
         11
19.75     9
19.65     8
20.05     8
19.9      8
19.55     7
45.3      7
20.15     6
20.25     6
Name: count, dtype: int64

=== TOTAL_CHARGES CONVERSION CHECK ===
Original dtype: object
Converted dtype: float64
Nulls after conversion: 11

Rows where total_charges became NaN after conversion:
      customerid  tenure_months  monthly_charges total_charges churn_label  \
2234  4472-LVYGI              0            52.55                        No   
2438  3115-CZMZD              0            20.25                        No   
2568  5709-LVOEQ              0            80.85                        No   
2667  4367-NUYAO              0            25.75                        No   
2856  1371-DWPAZ              0            56.05                        No   
4331  7644-OMVMY              0            19.85                        No   
4687  3213-VVOLG              0            25.35                 

In [49]:
    print("\n=== CATEGORY LEVEL REVIEW FOR LOW-CARDINALITY OBJECT COLUMNS ===")
    low_card_cols = [col for col in object_cols if df[col].nunique(dropna=False) <= 10]
    for col in low_card_cols:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False))


=== CATEGORY LEVEL REVIEW FOR LOW-CARDINALITY OBJECT COLUMNS ===

--- country ---
country
United States    7043
Name: count, dtype: int64

--- state ---
state
California    7043
Name: count, dtype: int64

--- gender ---
gender
Male      3555
Female    3488
Name: count, dtype: int64

--- senior_citizen ---
senior_citizen
No     5901
Yes    1142
Name: count, dtype: int64

--- partner ---
partner
No     3641
Yes    3402
Name: count, dtype: int64

--- dependents ---
dependents
No     5416
Yes    1627
Name: count, dtype: int64

--- phone_service ---
phone_service
Yes    6361
No      682
Name: count, dtype: int64

--- multiple_lines ---
multiple_lines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

--- internet_service ---
internet_service
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

--- online_security ---
online_security
No                     3498
Yes                    2019
No internet servic

In [50]:
id_cols = [col for col in ["customerid"] if col in df.columns]
id_cols

['customerid']

In [54]:
    id_cols = [col for col in ["customerid"] if col in df.columns]
    target_cols = [col for col in ["churn_value", "churn_label"] if col in df.columns]
    leakage_review_cols = [
        col for col in ["churn_reason", "churn_score", "cltv"] if col in df.columns
    ]
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = object_cols.copy()


In [55]:
    print("\n=== COLUMN GROUPING FOR NEXT STEPS ===")
    print("ID columns:", id_cols)
    print("Target columns:", target_cols)
    print("Leakage review columns:", leakage_review_cols)
    print("Numeric columns:", numeric_cols)
    print("Categorical columns:", categorical_cols)



=== COLUMN GROUPING FOR NEXT STEPS ===
ID columns: ['customerid']
Target columns: ['churn_value', 'churn_label']
Leakage review columns: ['churn_reason', 'churn_score', 'cltv']
Numeric columns: ['count', 'zip_code', 'latitude', 'longitude', 'tenure_months', 'monthly_charges', 'churn_value', 'churn_score', 'cltv', 'total_charges_clean']
Categorical columns: ['customerid', 'country', 'state', 'city', 'lat_long', 'gender', 'senior_citizen', 'partner', 'dependents', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing', 'payment_method', 'total_charges', 'churn_label', 'churn_reason']
